# Assignment 1
**Credits**: Federico Ruggeri, Eleonora Mancini, Paolo Torroni

**Keywords**: Sexism Detection, Multi-class Classification, RNNs, Transformers, Huggingface



# Contact
For any doubt, question, issue or help, you can always contact us at the following email addresses:

Teaching Assistants:

- Federico Ruggeri -> federico.ruggeri6@unibo.it
- Eleonora Mancini -> e.mancini@unibo.it

Professor:
- Paolo Torroni -> p.torroni@unibo.it

# Introduction
You are asked to address the [EXIST 2023 Task 2](https://clef2023.clef-initiative.eu/index.php?page=Pages/labs.html#EXIST) on sexism detection.

## Problem Definition

This task aims to categorize the sexist messages according to the intention of the author in one of the following categories: (i) direct sexist message, (ii) reported sexist message and (iii) judgemental message.

### Examples:

#### DIRECT 
The intention was to write a message that is sexist by itself or incites to be sexist, as in:

''*A woman needs love, to fill the fridge, if a man can give this to her in return for her services (housework, cooking, etc), I don’t see what else she needs.*''

#### REPORTED
The intention is to report and share a sexist situation suffered by a woman or women in first or third person, as in:

''*Today, one of my year 1 class pupils could not believe he’d lost a race against a girl.*''

#### JUDGEMENTAL
The intention was to judge, since the tweet describes sexist situations or behaviours with the aim of condemning them.

''*As usual, the woman was the one quitting her job for the family’s welfare…*''

# [Task 1 - 1.0 points] Corpus

We have preparared a small version of EXIST dataset in our dedicated [Github repository](https://github.com/lt-nlp-lab-unibo/nlp-course-material/tree/main/2025-2026/Assignment%201/data).

Check the `A1/data` folder. It contains 3 `.json` files representing `training`, `validation` and `test` sets.


### Dataset Description
- The dataset contains tweets in both English and Spanish.
- There are labels for multiple tasks, but we are focusing on **Task 2**.
- For Task 2, labels are assigned by six annotators.
- The labels for Task 2 represent whether the tweet is non-sexist ('-') or its sexist intention ('DIRECT', 'REPORTED', 'JUDGEMENTAL').







### Example

```
    "203260": {
        "id_EXIST": "203260",
        "lang": "en",
        "tweet": "ik when mandy says “you look like a whore” i look cute as FUCK",
        "number_annotators": 6,
        "annotators": ["Annotator_473", "Annotator_474", "Annotator_475", "Annotator_476", "Annotator_477", "Annotator_27"],
        "gender_annotators": ["F", "F", "M", "M", "M", "F"],
        "age_annotators": ["18-22", "23-45", "18-22", "23-45", "46+", "46+"],
        "labels_task1": ["YES", "YES", "YES", "NO", "YES", "YES"],
        "labels_task2": ["DIRECT", "DIRECT", "REPORTED", "-", "JUDGEMENTAL", "REPORTED"],
        "labels_task3": [
          ["STEREOTYPING-DOMINANCE"],
          ["OBJECTIFICATION"],
          ["SEXUAL-VIOLENCE"],
          ["-"],
          ["STEREOTYPING-DOMINANCE", "OBJECTIFICATION"],
          ["OBJECTIFICATION"]
        ],
        "split": "TRAIN_EN"
      }
    }
```

### Instructions
1. **Download** the `A1/data` folder.
2. **Load** the three JSON files and encode them as ``pandas.DataFrame``.
3. **Aggregate labels** for Task 2 using majority voting and store them in a new dataframe column called `label`. Items without a clear majority will be removed from the dataset.
4. **Filter the DataFrame** to keep only rows where the `lang` column is `'en'`.
5. **Remove unwanted columns**: Keep only `id_EXIST`, `lang`, `tweet`, and `label`.
6. **Encode the `label` column**: Use the following mapping

```
{
    '-': 0,
    'DIRECT': 1,
    'JUDGEMENTAL': 2,
    'REPORTED': 3
}
```

In [106]:
import json
import pandas as pd
from collections import Counter

# Load and encode the jsons
train_df = pd.DataFrame(json.load(open('./data/training.json'))).transpose()
test_df = pd.DataFrame(json.load(open('./data/test.json'))).transpose()
val_df = pd.DataFrame(json.load(open('./data/validation.json'))).transpose()
dfs = [train_df, val_df, test_df]

print("Before:", train_df.shape, val_df.shape, test_df.shape)

for i, df in enumerate(dfs):
    # Aggregate labels by majority vote, only keep rows with unique majority
    majority_labels = []
    for labels in df["labels_task2"]:
        c = Counter(labels).most_common(2)
        if len(c)==1:
            majority_labels.append(c[0][0])
        elif c[1][1] < c[0][1]:
            majority_labels.append(c[0][0])
        else:
            majority_labels.append(None)
    df["label"] = majority_labels
    df = df.dropna(subset=["label"]).reset_index(drop=True)

    # Filter rows with language "en"
    df = df[df["lang"]=="en"].reset_index(drop=True)
    df = df[["id_EXIST", "lang", "tweet", "label"]]

    # Encode labels as integers
    label_mapping = {"-":0, "DIRECT":1, "JUDGEMENTAL":2, "REPORTED":3}
    df["label"] = df["label"].map(label_mapping)

    dfs[i] = df # NOTE: we have to do this otherwise the changes won't persist


train_df, val_df, test_df = dfs
print("After: ", train_df.shape, val_df.shape, test_df.shape)

Before: (6920, 11) (726, 11) (312, 11)
After:  (2873, 4) (150, 4) (280, 4)


# [Task2 - 0.5 points] Data Cleaning
In the context of tweets, we have noisy and informal data that often includes unnecessary elements like emojis, hashtags, mentions, and URLs. These elements may interfere with the text analysis.



### Instructions
- **Remove emojis** from the tweets.
- **Remove hashtags** (e.g., `#example`).
- **Remove mentions** such as `@user`.
- **Remove URLs** from the tweets.
- **Remove special characters and symbols**.
- **Remove specific quote characters** (e.g., curly quotes).
- **Perform lemmatization** to reduce words to their base form.

In [107]:
import re
import nltk
from nltk import pos_tag, word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

# Download required NLTK data
# NOTE: this is mostly from the lab, mayeb it's not the most efficient way
nltk.download('omw-1.4')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

lemmatizer = WordNetLemmatizer()

def pos2wordnet_tag(treebank_tag: str) -> str:
    match treebank_tag[0]:
        case "J":   return wordnet.ADJ
        case "V":   return wordnet.VERB
        case "N":   return wordnet.NOUN
        case "R":   return wordnet.ADV
        case _:     return wordnet.NOUN

def token_lemma(text: str) -> str:
    """
    Tokenize the text, then lemmatize the tokens and then merge the lemmatized tokens into a text
    """
    tokens = word_tokenize(text)    # usese recommended NLTK tokenizer
    tagged_tokens = pos_tag(tokens)
    lemmatized_tokens = [
        lemmatizer.lemmatize(tok.lower(), pos2wordnet_tag(pos))
        for tok, pos in tagged_tokens
    ]
    return " ".join(lemmatized_tokens)

def clean_text(text, patterns):
    # Remove patterns
    for pattern in patterns:
        text = pattern.sub('', text)
        
    # EXTRA: post-processing
    text = re.sub(r'\s+', ' ', text)             # multiple spaces => single space
    text = re.sub(r'\s([.,;!?])', r'\1', text)   # remove space before punctuation
    text = re.sub(r'^[.,;!?\s]+', '', text)      # remove punctuation/spaces at the beginning
    text = re.sub(r'[\s]+$', '', text)           # remove spaces at the end
    text = re.sub(r'([.,;!?]){2,}', r'\1', text) # multiple punctuations => single punctuation
    
    # Tokenize, lemmatize and merge
    return token_lemma(text)

[nltk_data] Downloading package omw-1.4 to /Users/pelle/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/pelle/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/pelle/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/pelle/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [108]:
# NOTE: THIS IS THE PREVIOUS VERSION, IT IS NOT EXECUTED
# I KEPT THIS CELL SINCE I ANNOTED SOME TRICKY CHARS

if False:
    # NOTE: 1 big regex for all patterns would be more efficient, but here we prefer to keep them separate for clarity
    patterns = [
        # 1) Emojis
        # NOTE: there is the emoji library to do it, but here i use only regex to match lessons
        # Missing: 🤔🤣 :) XO 🧋🤷🥺🤙🤝🤩🥴🤮🤢🤯🤫🤩🥳🤦‍🤣🥰🧳🤡🥱 :'''') 🤬🧵🧿 ^.^ 🥶🥦🤠🦕🤪 x') 🤞🤕🧍‍🧎‍🪑🥲🤤🧚‍🧿 xD 🩸🤍🤺🦸🦉🥵🤚⏰🤘🤨🤵🤗🫵🧛
        re.compile(
            "["               
            u"\U0001F600-\U0001F64F"  # Emoticons
            u"\U0001F300-\U0001F5FF"  # Symbols & Pictographs
            u"\U0001F680-\U0001F6FF"  # Transport & Map Symbols
            u"\U0001F1E0-\U0001F1FF"  # Flags (iOS)
            u"\U00002702-\U000027B0"
            u"\U000024C2-\U0001F251"
            "]+", flags=re.UNICODE
        ),
        
        # 4) URLs
        # NOTE: if we do this before removing hashtags we avoid broken urls, as in retriving the tweets some urls were appended to hashtags
        re.compile(r'https?://[^\s]+'),
        
        # 2) Hashtags
        re.compile(r'#\w+'),
        
        # 3) Mentions
        re.compile(r'@\w+'),
        
        # 5) Special characters and symbols
        # &gt &lt;3&lt;3 &amp; ↕ 10%- ⁦ ⁩ ʸᵒᵘ ˡᵒᵒᵏ ˡⁱᵏᵉ ᵃ ᵇⁱᵗᶜʰ /lh
        
        # 6) Specific quotes characters, i assume not the regular ones (e.g. " ' )
        re.compile(r'[“”‘’«»`]')
        # ´
    ]

    for i, df in enumerate(dfs):
        # TODO: dopo scommenta
        # df['cleaned_tweet'] = df["tweet"].apply(lambda text: clean_text(text, patterns))
        # dfs[i] = df    
        
        # DEBUG: print cleaned tweets
        for txt in df['tweet']: 
            for pattern in patterns: txt = pattern.sub('', txt)
            print(txt, "\n")
        

    # display(train_df[["tweet", "cleaned_tweet"]])

In [109]:
# NOTE: 1 big regex for all patterns would be more efficient, but here we prefer to keep them separate for clarity

patterns = [
    # 1) Emoji
    # 1.1) Textual emoji
    re.compile(
        r'(?::|;|=|x)[\'\-]?(?:\)|D|\(|P|p|O|o|\||\\|/|\*|\[|\]){1,}|'  # classical order
        r'(?:\)|D|\(|P|p|O|o){1,}(?::|;|=)',  # inverted order
        re.IGNORECASE
    ),
    # 1.2) Unicoed emoji
    re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # Emoticons
        u"\U0001F300-\U0001F5FF"  # Symbols & Pictographs
        u"\U0001F680-\U0001F6FF"  # Transport & Map
        u"\U0001F700-\U0001F77F"  # Alchemical Symbols
        u"\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
        u"\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
        u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        u"\U0001FA00-\U0001FA6F"  # Chess Symbols
        u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        u"\U00002702-\U000027B0"  # Dingbats
        u"\U000024C2-\U0001F251"
        u"\U0001F1E0-\U0001F1FF"  # Flags
        u"\U0001F900-\U0001F9FF"  # Supplemental Symbols
        u"\U0001FA70-\U0001FAFF"  # Extended pictographs
        u"\U00002600-\U000026FF"  # Miscellaneous symbols
        u"\U00002700-\U000027BF"  # Dingbats
        "]+",
        flags=re.UNICODE
    ),
    
    # 4) URLs
    # NOTE: if we do this before removing hashtags we avoid broken urls, as in retriving the tweets some urls were appended to hashtags
    re.compile(r'https?[^\s]+'), # NB: don't put also :// since they are removed as text emoji
    
    # 2) Hashtags
    re.compile(r'#\w+'),
    
    # 3) Mentions
    re.compile(r'@\w+'),
    
    # 5) Special characters and symbols
    # 5.1) HTML entities
    re.compile(r'&[a-zA-Z]+;|&#\d+;|&[a-zA-Z]+'),
    # 5.2) Strance unicode chars
    re.compile(r'[↕↔→←↑↓⇒⇐⇑⇓⇔⟵⟶⟷⤴⤵]'), # it is used once, probably overfitting
    re.compile(r'[ᵃᵇᶜᵈᵉᶠᵍʰⁱʲᵏˡᵐⁿᵒᵖʳˢᵗᵘᵛʷˣʸᶻᴬᴮᴰᴱᴳᴴᴵᴶᴷᴸᴹᴺᴼᴾᴿᵀᵁⱽᵂ]+'),  # NOTE: there is a sentece where they are useful
    # 5.3) Fractions and percentages
    re.compile(r'\d+%[-]?(?!\w)'), # NOTE: maybe it is overfit, but there is a sentence with 10 times 10%- which is weird
    
    # 6) Specific quotes characters, i assume not the regular ones (e.g. " ' )
    re.compile(r'[“”‘’«»`´]'), 
]

for i, df in enumerate(dfs):
    df['cleaned_tweet'] = df["tweet"].apply(lambda text: clean_text(text, patterns))
    dfs[i] = df    

display(train_df[["tweet", "cleaned_tweet"]])

,tweet,cleaned_tweet
0,FFS! How about laying the blame on the bastard...,ffs ! how about lay the blame on the bastard w...
1,Writing a uni essay in my local pub with a cof...,write a uni essay in my local pub with a coffe...
2,@UniversalORL it is 2021 not 1921. I dont appr...,it be 2021 not 1921 . i dont appreciate that o...
3,@GMB this is unacceptable. Use her title as yo...,this be unacceptable . use her title a you do ...
4,‘Making yourself a harder target’ basically bo...,make yourself a hard target basically boil dow...
...,...,...
2868,@ShefVaidya Ma'am if I say that you look like ...,"ma'am if i say that you look like a whore , wo..."
2869,idk why y’all bitches think having half your a...,idk why yall bitch think have half your as han...
2870,This has been a part of an experiment with @Wo...,this have be a part of an eeriment with . what...
2871,"""Take me already"" ""Not yet. You gotta be ready...",`` take me already '' `` not yet . you get ta ...


In [110]:
# DEBUG: print cleaned tweets
for i, df in enumerate(dfs):    
    for txt in df['cleaned_tweet']: 
        # for pattern in patterns: txt = pattern.sub('', txt)
        print(txt, "\n")

ffs ! how about lay the blame on the bastard who murder her ? novel idea , i know . 

write a uni essay in my local pub with a coffee . random old man keep ask me drunk question when i 'm try to concentrate end with `` good luck , but you 'll just end up get married and not use it anyway '' . be alive and well 

it be 2021 not 1921 . i dont appreciate that on two ride by myself your team member look behind me and ask the man behind how many in my party . not impressed 

this be unacceptable . use her title a you do for all the men interview . she be , in fact , senior to angus 

make yourself a hard target basically boil down to make sure they target someone else 

accord to a customer i have plenty of time to go spent the stirling coin he want to pay me with , in derry . `` just like any other woman , i 'm sure of it . '' in retail . 

so only 'blokes ' drink beer ? sorry , but if you be n't a 'bloke ' you drink wine apparently . alive and well in 

new to the shelf this week - look f

In [111]:
# DEBUG: print original tweets for comaprison
for i, df in enumerate(dfs):    
    for txt in df['tweet']: 
        # for pattern in patterns: txt = pattern.sub('', txt)
        print(txt, "\n")

FFS! How about laying the blame on the bastard who murdered her? Novel idea, I know. https://t.co/GI5B45THvJ 

Writing a uni essay in my local pub with a coffee. Random old man keeps asking me drunk questions when I'm trying to concentrate &amp; ends with "good luck, but you'll just end up getting married and not use it anyway". #EverydaySexism is alive and well 🙃 

@UniversalORL it is 2021 not 1921. I dont appreciate that on two rides by myself your team member looked behind me and asked the man behind how many in my party. Not impressed #everydaysexism 

@GMB this is unacceptable. Use her title as you did for all the men interviewed. She is, in fact, senior to Angus #everydaysexism @TheWomensOrg https://t.co/JZF3a5E4eX 

‘Making yourself a harder target’ basically boils down to ‘make sure they target someone else’ 🙃 https://t.co/VOpu09YAj6 

According to a customer I have plenty of time to go spent the Stirling coins he wants to pay me with, in Derry. "Just like any other woman, I'm 

# [Task 3 - 0.5 points] Text Encoding
To train a neural sexism classifier, you first need to encode text into numerical format.




### Instructions

* Embed words using **GloVe embeddings**.
* You are **free** to pick any embedding dimension.





### What about OOV tokens?
   * All the tokens in the **training** set that are not in GloVe **must** be added to the vocabulary.
   * For the remaining tokens (i.e., OOV in the validation and test sets), you have to assign them a **special token** (e.g., ``<UNK>``) and a **static** embedding.
   * You are **free** to define the static embedding using any strategy (e.g., random, neighbourhood, etc...)



### More about OOV

For a given token:

* **If in train set**: add to vocabulary and assign an embedding (use GloVe if token in GloVe, custom embedding otherwise).
* **If in val/test set**: assign special token if not in vocabulary and assign custom embedding.

Your vocabulary **should**:

* Contain all tokens in train set; or
* Union of tokens in train set and in GloVe $\rightarrow$ we make use of existing knowledge!

# [Task 4 - 1.0 points] Model definition

You are now tasked to define your sexism classifier.




### Instructions

* **Baseline**: implement a Bidirectional LSTM with a Dense layer on top.

* **Stacked**: add an additional Bidirectional LSTM layer to the Baseline model.

**Note**: You are **free** to experiment with hyper-parameters.

### Token to embedding mapping

You can follow two approaches for encoding tokens in your classifier.

### Work directly with embeddings

- Compute the embedding of each input token
- Feed the mini-batches of shape ``(batch_size, # tokens, embedding_dim)`` to your model

### Work with Embedding layer

- Encode input tokens to token ids
- Define a Embedding layer as the first layer of your model
- Compute the embedding matrix of all known tokens (i.e., tokens in your vocabulary)
- Initialize the Embedding layer with the computed embedding matrix
- You are **free** to set the Embedding layer trainable or not

In [112]:
embedding = tf.keras.layers.Embedding(input_dim=vocab_size,
                                      output_dim=embedding_dimension,
                                      weights=[embedding_matrix],
                                      mask_zero=True,                   # automatically masks padding tokens
                                      name='encoder_embedding')

NameError: name 'tf' is not defined

# [Task 5 - 1.0 points] Training and Evaluation

You are now tasked to train and evaluate the Baseline and Stacked models.



### Instructions

* Pick **at least** three seeds for robust estimation.
* Train **all** models on the train set.
* Evaluate **all** models on the validation and test sets.
* Compute macro F1-score, precision, and recall metrics on the validation set.
* Report average and standard deviation measures over seeds for each metric.
* Pick the **best** performing model according to the observed validation set performance (use macro F1-score).

# [Task 6 - 1.0 points] Transformers

In this section, you will use a transformer model specifically trained for hate speech detection, namely [Twitter-roBERTa-base for Hate Speech Detection](https://huggingface.co/cardiffnlp/twitter-roberta-base-hate).




### Relevant Material
- Tutorial 3

### Instructions
- **Load the Tokenizer and Model**

- **Preprocess the Dataset**:
   You will need to preprocess your dataset to prepare it for input into the model. Tokenize your text data using the appropriate tokenizer and ensure it is formatted correctly.

- **Train the Model**:
   Use the `Trainer` to train the model on your training data.

- **Evaluate the Model on the Test Set** using the same metrics used for LSTM-based models.

# [Task 7 - 0.5 points] Error Analysis

After evaluating the model, perform a brief error analysis on the **test set**:

### Instructions

 - Review the results and identify common errors.

 - Summarize your findings regarding the errors and their impact on performance (e.g. but not limited to Out-of-Vocabulary (OOV) words, data imbalance, and performance differences between the custom model and the transformer...)
 - Suggest possible solutions to address the identified errors.

# [Task 8 - 0.5 points] Report

Wrap up your experiment in a short report (up to 2 pages).

### Instructions

* Use the NLP course report template.
* Summarize each task in the report following the provided template.

### Recommendations

The report is **not a copy-paste** of graphs, tables, and command outputs.

* Summarize classification performance in Table format.
* **Do not** report command outputs or screenshots.
* Report learning curves in Figure format.
* The error analysis section should summarize your findings.


# Submission

* **Submit** your report in PDF format.
* **Submit** your python notebook.
* Make sure your notebook is **well organized**, with no temporary code, commented sections, tests, etc...
* You can upload **model weights** in a cloud repository and report the link in the report.

## Bonus Points
Bonus points are arbitrarily assigned based on significant contributions such as:
- Outstanding error analysis
- Masterclass code organization
- Suitable extensions

**Note**: bonus points are only assigned if all task points are attributed (i.e., 6/6).

**Possible Suggestions for Bonus Points:**
- **Try other preprocessing strategies**: e.g., but not limited to, explore techniques tailored specifically for tweets or  methods that are common in social media text.
- **Experiment with other custom architectures or models from HuggingFace**
- **Explore Spanish tweets**: e.g., but not limited to, leverage multilingual models to process Spanish tweets and assess their performance compared to monolingual models.

# FAQ

Please check this frequently asked questions before contacting us

### Trainable Embeddings

You are **free** to define a trainable or non-trainable Embedding layer to load the GloVe embeddings.

### Model architecture

You **should not** change the architecture of a model (i.e., its layers).

However, you are **free** to play with their hyper-parameters.


### Neural Libraries

You are **free** to use any library of your choice to implement the networks (e.g., Keras, Tensorflow, PyTorch, JAX, etc...)

### Robust Evaluation

Each model is trained with at least 3 random seeds.

Task 5 requires you to compute the average performance over the 3 seeds and its corresponding standard deviation.

### Expected Results

Task 2 leaderboard reports around 40-50 F1-score.
However, note that they perform a hierarchical classification.

That said, results around 30-40 F1-score are **expected** given the task's complexity.

### Model Selection for Analysis

To carry out the error analysis you are **free** to either

* Pick examples or perform comparisons with an individual seed run model (e.g., Baseline seed 1337)
* Perform ensembling via, for instance, majority voting to obtain a single model.

### Error Analysis

Some topics for discussion include:
   * Precision/Recall curves.
   * Confusion matrices.
   * Specific misclassified samples.


# The End

Feel free to reach out for questions/doubts!